In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import os

# Para pose
import mediapipe as mp

ROOT = Path(".")
DATA = ROOT / "Playground" / "data"
DOWNLOADS = ROOT / "Playground" / "Downloads"

df_videos = pd.read_csv(DATA / "videos.csv")
df_videos.head()



ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
def load_video_12fps(video_path, target_fps=12):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30

    frame_step = max(int(round(fps / target_fps)), 1)
    frames = []
    idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % frame_step == 0:
            frames.append(frame)
        idx += 1

    cap.release()
    return frames


In [ ]:
from ultralytics import YOLO

model_yolo = YOLO("yolov8n.pt")  # ultraligero y rápido

def run_yolo(frame):
    results = model_yolo(frame, verbose=False)
    dets = []
    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            if cls == 0:  # persona
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                dets.append((x1, y1, x2, y2))
    return dets


In [ ]:
class SimpleTracker:
    def __init__(self):
        self.next_id = 0

    def update(self, detections):
        tracks = []
        for d in detections:
            x1,y1,x2,y2 = d
            tracks.append((self.next_id, x1,y1,x2,y2))
            self.next_id += 1
        return tracks

tracker = SimpleTracker()



In [ ]:
mp_pose = mp.solutions.pose
pose_estimator = mp_pose.Pose(static_image_mode=True)

def run_pose(frame, bbox):
    x1,y1,x2,y2 = map(int, bbox)
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return np.zeros((17,2), dtype=np.float32)

    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    res = pose_estimator.process(crop_rgb)

    kps = np.zeros((17,2), dtype=np.float32)

    if res.pose_landmarks:
        for i,lm in enumerate(res.pose_landmarks.landmark[:17]):
            kps[i,0] = lm.x
            kps[i,1] = lm.y

    return kps


In [ ]:
def normalize_skeleton(kps, eps=1e-6):
    kps = kps.astype(np.float32).copy()
    
    pelvis = (kps[11] + kps[12]) / 2
    kps -= pelvis
    
    shoulders_center = (kps[5] + kps[6]) / 2
    torso_len = np.linalg.norm(shoulders_center) + eps
    kps /= torso_len

    return kps


Procesando: columpioscam2-2024-12-17 19 -> Downloads/columpioscam2-2024-12-17 19


In [ ]:
def extract_skeletons(frames):
    results = []

    for i, frame in enumerate(frames):
        dets = run_yolo(frame)
        tracks = tracker.update(dets)

        for tid,x1,y1,x2,y2 in tracks:
            kps = run_pose(frame, (x1,y1,x2,y2))
            kps_norm = normalize_skeleton(kps)

            results.append({
                "local_frame": i,
                "track_id": tid,
                "kps": kps_norm
            })

    return results


In [ ]:
def build_windows(skeletons, T=48, K_max=4):
    if not skeletons:
        return np.empty((0,T,K_max,17,2), dtype=np.float32)

    by_track = {}
    for s in skeletons:
        by_track.setdefault(s["track_id"], []).append(s)

    for tr in by_track.values():
        tr.sort(key=lambda x: x["local_frame"])

    all_frames = sorted([s["local_frame"] for s in skeletons])
    min_f, max_f = all_frames[0], all_frames[-1]

    windows = []
    start = min_f
    stride = T // 2

    while start + T - 1 <= max_f:
        end = start + T

        w = np.zeros((T, K_max, 17, 2), dtype=np.float32)

        track_scores = []
        for tid, tr in by_track.items():
            n = sum(start <= x["local_frame"] < end for x in tr)
            track_scores.append((n, tid))

        track_scores.sort(reverse=True)
        selected = [tid for n,tid in track_scores[:K_max] if n > 0]

        for k, tid in enumerate(selected):
            tr = by_track[tid]
            frame2kps = {x["local_frame"]: x["kps"] for x in tr}

            for i,f in enumerate(range(start,end)):
                if f in frame2kps:
                    w[i,k] = frame2kps[f]

        windows.append(w)
        start += stride

    return np.stack(windows, axis=0)


In [ ]:
OUT_NPY = DATA / "npy"
OUT_NPY.mkdir(exist_ok=True)

def local_video_path(row):
    return DOWNLOADS / os.path.basename(row["blob_path"])

def process_scene(row):
    vid_id = row["video_id"]
    vpath = local_video_path(row)

    print("Procesando:", vid_id, "->", vpath)

    frames = load_video_12fps(vpath)
    skels = extract_skeletons(frames)
    windows = build_windows(skels, T=48, K_max=4)

    for i,w in enumerate(windows):
        out = OUT_NPY / f"{vid_id}_win{i:03d}.npy"
        np.save(out, w.astype(np.float32))
        print("✓ Guardado:", out)


In [ ]:
df_sample = df_videos.head(1)
for _, row in df_sample.iterrows():
    process_scene(row)
